#### EDA 10: Joined order-level table
**Goal:** join all 8 tables into one table with **one row per order** (99,441 rows), resolve the cross-table checks collected in the earlier notebooks, and save the result as one Excel file.

**Rule for every join:** the right-hand table must be one row per key before the join, and the row count after the join must still be 99,441. The notebook stops with an error otherwise.

**Flow:** Load → Base (orders) → Customers → Items and products → Sellers → Payments → Geolocation → Reviews → Derived columns and flags → Missingness vs target → Data dictionary → Save to Excel → Summary

**Output:** `data/processed/orders_joined.xlsx` (sheets: `read_me`, `orders_joined`, `data_dictionary`, `join_log`, `checks`).

#### Load the raw tables
**Question:** are all 8 raw tables loaded with the right dates?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import unicodedata

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data
OUT_DIR = Path("../../data/processed")            # the Excel file is saved here
OUT_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW / "olist_orders_dataset.csv",
                     parse_dates=["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                                  "order_delivered_customer_date", "order_estimated_delivery_date"])
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_creation_date", "review_answer_timestamp"])
products = pd.read_csv(RAW / "olist_products_dataset.csv").rename(columns={"product_name_lenght": "product_name_length",
                                                                           "product_description_lenght": "product_description_length"})
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
geo = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
cat_en = pd.read_csv(RAW / "product_category_name_translation.csv")

pd.Series({"orders": len(orders), "customers": len(customers), "items": len(items), "payments": len(payments),
           "reviews": len(reviews), "products": len(products), "sellers": len(sellers), "geo": len(geo),
           "cat_en": len(cat_en)}, name="rows").to_frame()

,rows
orders,99441
customers,99441
items,112650
payments,103886
reviews,99224
products,32951
sellers,3095
geo,1000163
cat_en,71


- All 9 raw tables load with the row counts seen in the single-table notebooks (orders 99,441, items 112,650, payments 103,886, reviews 99,224, products 32,951, sellers 3,095, geolocation 1,000,163, customers 99,441, translation 71).

#### Step 0: base table (orders)
**Question:** what is the base, and how do joins get logged?

In [2]:
assert orders["order_id"].is_unique               # the grain: one row per order
base = orders.copy()
N_ORDERS = len(base)

join_log = []     # one line per join: sizes and matches
checks = []       # one line per cross-table check: result and note


def log_join(step, right_table, right_rows, rows_after, matched, note=""):
    """Record a join and stop if the row count changed."""
    assert rows_after == N_ORDERS, f"{step}: row count changed to {rows_after}"
    join_log.append({"step": step, "right_table": right_table, "right_rows_used": right_rows,
                     "rows_after_join": rows_after, "matched_orders": matched,
                     "unmatched_orders": N_ORDERS - matched, "note": note})


def add_check(area, check, result, note=""):
    checks.append({"area": area, "check": check, "result": result, "note": note})


print("base rows:", N_ORDERS)

base rows: 99441


- `order_id` is unique, so the base has 99,441 rows. This number is the grain, and every join below must keep it.
- Each join is logged (sheet `join_log`) and stops the notebook if the row count changes. Pandas `validate=` is also used, so a join that is not one-to-one or many-to-one raises an error.

#### Step 1: customers
**Question:** does each order have exactly one customer row?

In [3]:
assert customers["customer_id"].is_unique         # one row per customer_id

before = len(base)
base = base.merge(customers, on="customer_id", how="left", validate="one_to_one")
matched = base["customer_unique_id"].notna().sum()
log_join("1 customers", "customers", len(customers), len(base), matched, "key customer_id, one to one")

print("orders whose customer_id is in customers:", orders["customer_id"].isin(customers["customer_id"]).mean())
print("customer rows whose customer_id is in orders:", customers["customer_id"].isin(orders["customer_id"]).mean())
print("rows before / after:", before, "/", len(base))
base[["order_id", "customer_unique_id", "customer_zip_code_prefix", "customer_state"]].head(3)

orders whose customer_id is in customers: 1.0
customer rows whose customer_id is in orders: 1.0
rows before / after: 99441 / 99441


,order_id,customer_unique_id,customer_zip_code_prefix,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,3149,SP
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,47813,BA
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,75265,GO


- Every order has a customer row and every customer row has an order (both ratios are 1.0). The join is one-to-one and keeps 99,441 rows.

#### Step 2: items and products
Items have several lines per order, so they are reduced to one row per order before the join.

**Question 2a:** do all item lines point to real orders, products and sellers?

In [4]:
print("item order_id in orders     :", items["order_id"].isin(orders["order_id"]).mean())
print("item product_id in products :", items["product_id"].isin(products["product_id"]).mean())
print("item seller_id in sellers   :", items["seller_id"].isin(sellers["seller_id"]).mean())
print("sellers without any item    :", (~sellers["seller_id"].isin(items["seller_id"])).sum())
print("products without any item   :", (~products["product_id"].isin(items["product_id"])).sum())

add_check("items", "item lines with an order_id missing in orders", int((~items["order_id"].isin(orders["order_id"])).sum()))
add_check("items", "item lines with a product_id missing in products", int((~items["product_id"].isin(products["product_id"])).sum()))
add_check("items", "item lines with a seller_id missing in sellers", int((~items["seller_id"].isin(sellers["seller_id"])).sum()))
add_check("items", "sellers without any item line", int((~sellers["seller_id"].isin(items["seller_id"])).sum()))

item order_id in orders     : 1.0
item product_id in products : 1.0
item seller_id in sellers   : 1.0
sellers without any item    : 0
products without any item   : 0


- All item lines point to a real order, product and seller (ratios 1.0). No orphan lines.
- Every seller and every product appears in at least one item line (0 without items), so the sellers and products tables hold no unused records.

**Question 2b:** which orders have no item lines?

In [5]:
no_items = orders[~orders["order_id"].isin(items["order_id"])]
print("orders without items:", len(no_items))
display(no_items["order_status"].value_counts().rename("n_orders").to_frame())
print("of these never approved:", no_items["order_approved_at"].isna().sum())
add_check("items", "orders without any item line", len(no_items), "mostly unavailable and canceled")

orders without items: 775


,n_orders
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


of these never approved: 146


- 775 orders have no item lines: `unavailable` 603, `canceled` 164, `created` 5, `invoiced` 2 and `shipped` 1. 146 of them were never approved.
- These orders never reached a seller, so they get empty seller, category and price fields. They are kept and flagged (see the scope flags below).

**Question 2c:** how many rows would a direct merge create? (the grain proof)

In [6]:
naive = orders.merge(items, on="order_id", how="left")
print("orders                          :", len(orders))
print("direct merge orders + item lines:", len(naive), "-> extra rows:", len(naive) - len(orders))
add_check("items", "rows after a direct merge of item lines onto orders", len(naive), f"{len(naive) - len(orders)} extra rows against {len(orders)} orders")

orders                          : 99441
direct merge orders + item lines: 113425 -> extra rows: 13984


- A direct merge of item lines onto orders gives 113,425 rows, which is 13,984 more than the 99,441 orders. A count or an average on that table would count orders several times.
- The aggregated merge below keeps 99,441 rows.

**Question 2d:** do product categories have an English name?

In [7]:
cat_coverage = products["product_category_name"].dropna().drop_duplicates()
no_english = sorted(set(cat_coverage) - set(cat_en["product_category_name"]))
print("product categories                    :", len(cat_coverage))
print("categories without an English name    :", len(no_english), no_english)
print("translation names not used by products:", len(set(cat_en["product_category_name"]) - set(cat_coverage)))
add_check("products", "product categories without an English name", len(no_english), ", ".join(no_english))

# English names: the 5 spelling errors found in the translation notebook (eda_09) are fixed with an explicit list
EN_FIXES = {"fashio_female_clothing": "fashion_female_clothing", "costruction_tools_garden": "construction_tools_garden",
            "costruction_tools_tools": "construction_tools_tools", "home_confort": "home_comfort",
            "arts_and_craftmanship": "arts_and_craftsmanship"}
cat_en = cat_en.assign(product_category_name_english=cat_en["product_category_name_english"].replace(EN_FIXES))
print("English names corrected:", len(EN_FIXES), "| names still unique:", cat_en["product_category_name_english"].is_unique)
add_check("products", "English category names with a spelling error corrected", len(EN_FIXES), ", ".join(EN_FIXES))

# Product attributes on each item line (many lines to one product)
products["volume_cm3"] = products["product_length_cm"] * products["product_height_cm"] * products["product_width_cm"]
products["no_listing_info"] = products[["product_category_name", "product_name_length", "product_description_length",
                                        "product_photos_qty"]].isna().all(axis=1)
products_en = products.merge(cat_en, on="product_category_name", how="left", validate="many_to_one")
assert len(products_en) == len(products)

lines = items.merge(products_en[["product_id", "product_category_name", "product_category_name_english", "product_weight_g",
                                 "volume_cm3", "no_listing_info"]], on="product_id", how="left", validate="many_to_one")
assert len(lines) == len(items)
print("\nitem lines:", len(lines), "| lines whose product has no listing info:", lines["no_listing_info"].sum(),
      f"({lines['no_listing_info'].mean() * 100:.2f}%)")
print("orders with at least one such line:", lines.loc[lines["no_listing_info"], "order_id"].nunique())
add_check("products", "orders with a product that has no listing info", int(lines.loc[lines["no_listing_info"], "order_id"].nunique()))

product categories                    : 73
categories without an English name    : 2 ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
translation names not used by products: 0
English names corrected: 5 | names still unique: True



item lines: 112650 | lines whose product has no listing info: 1603 (1.42%)
orders with at least one such line: 1451


- 73 product categories exist and 2 have no English name: `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`. All 71 translation names are used.
- 1,603 item lines (1.42%) belong to products with no listing information. They sit in 1,451 orders.
- **Decision:** keep these lines. The English category stays empty for the 2 untranslated categories, and the Portuguese `primary_category` is always kept.
- 5 English names with spelling errors (found in the translation notebook, eda_09) are fixed with an explicit list. The raw file is not changed.

**Question 2e:** what does one row per order look like after reducing the lines?

In [8]:
# Value share of each seller and each category inside an order (price only)
seller_value = lines.groupby(["order_id", "seller_id"], as_index=False)["price"].sum()
seller_value["share"] = seller_value["price"] / seller_value.groupby("order_id")["price"].transform("sum")
primary_seller = (seller_value.sort_values(["order_id", "share", "seller_id"], ascending=[True, False, True])
                  .groupby("order_id").head(1).set_index("order_id")["seller_id"])

cat_value = lines.groupby(["order_id", "product_category_name"], as_index=False)["price"].sum()
primary_category = (cat_value.sort_values(["order_id", "price", "product_category_name"], ascending=[True, False, True])
                    .groupby("order_id").head(1).set_index("order_id")["product_category_name"])

# Shipping deadline of the earliest line, in days after the purchase
lines = lines.merge(orders[["order_id", "order_purchase_timestamp", "order_approved_at"]], on="order_id", how="left", validate="many_to_one")
lines["limit_days"] = (lines["shipping_limit_date"] - lines["order_purchase_timestamp"]).dt.total_seconds() / 86400

order_items = lines.groupby("order_id").agg(
    n_lines=("order_item_id", "size"),
    n_products=("product_id", "nunique"),
    n_sellers=("seller_id", "nunique"),
    n_categories=("product_category_name", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    total_weight_g=("product_weight_g", lambda s: s.sum(min_count=1)),
    total_volume_cm3=("volume_cm3", lambda s: s.sum(min_count=1)),
    weight_incomplete=("product_weight_g", lambda s: s.isna().any()),
    any_no_listing_info=("no_listing_info", "any"),
    limit_days_min=("limit_days", "min"),
)
order_items["freight_ratio"] = order_items["total_freight"] / order_items["total_price"]
order_items["primary_seller_id"] = primary_seller
order_items["primary_category"] = primary_category
order_items["primary_category_en"] = order_items["primary_category"].map(cat_en.set_index("product_category_name")["product_category_name_english"])

assert order_items.index.is_unique
print("order-level items table:", order_items.shape)

before = len(base)
base = base.merge(order_items, left_on="order_id", right_index=True, how="left", validate="one_to_one")
base["has_items"] = base["n_lines"].notna()
log_join("2 items + products", "items (reduced to one row per order)", len(order_items), len(base), int(base["has_items"].sum()),
         "key order_id; adds seller and category through the lines")
print("rows before / after:", before, "/", len(base), "| orders with items:", base["has_items"].sum())
order_items.head(3).round(2)

order-level items table: (98666, 15)
rows before / after: 99441 / 99441 | orders with items: 98666


,n_lines,n_products,n_sellers,n_categories,total_price,total_freight,total_weight_g,total_volume_cm3,weight_incomplete,any_no_listing_info,limit_days_min,freight_ratio,primary_seller_id,primary_category,primary_category_en
order_id,,,,,,,,,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,1,1,1,1,58.9,13.29,650.0,3528.0,False,False,6.03,0.23,48436dade18ac8b2bce089ec2a041202,cool_stuff,cool_stuff
00018f77f2f0320c557190d7a144bdd3,1,1,1,1,239.9,19.93,30000.0,60000.0,False,False,7.01,0.08,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,pet_shop
000229ec398224ef6ca0657da4fc703e,1,1,1,1,199.0,17.87,3050.0,14157.0,False,False,4.01,0.09,5b51032eddd242adc84c38acab88f23d,moveis_decoracao,furniture_decor


- The 112,650 lines reduce to 98,666 order rows. Merged onto the base, the result has exactly 99,441 rows, with 98,666 orders that have items.
- The table carries counts (`n_lines`, `n_products`, `n_sellers`, `n_categories`), values (`total_price`, `total_freight`, `freight_ratio`), declared weight and volume, `primary_seller_id` and `primary_category` (largest share of the order value) and the earliest shipping deadline.

**Question 2f:** is the shipping deadline believable against the purchase and approval dates?

In [9]:
display(lines["limit_days"].describe(percentiles=[0.01, 0.5, 0.95, 0.99]).round(2).to_frame("days from purchase to deadline"))
n_before_approval = (lines["shipping_limit_date"] < lines["order_approved_at"]).sum()
print("lines with deadline before the approval time :", n_before_approval)
print("lines with deadline before the purchase      :", (lines["limit_days"] < 0).sum())
print("lines with deadline more than 30 days out    :", (lines["limit_days"] > 30).sum())
add_check("items", "lines with a shipping deadline before the approval time", int(n_before_approval))
add_check("items", "lines with a shipping deadline before the purchase", int((lines["limit_days"] < 0).sum()))
add_check("items", "median days from purchase to shipping deadline", round(float(lines["limit_days"].median()), 2),
          "minimum " + str(round(float(lines["limit_days"].min()), 2)))

,days from purchase to deadline
count,112650.00
mean,6.64
std,7.07
min,2.00
1%,2.01
50%,6.01
95%,12.03
99%,19.01
max,1056.04


lines with deadline before the approval time : 127
lines with deadline before the purchase      : 0
lines with deadline more than 30 days out    : 239


- The deadline is 2.00 days after the purchase at the earliest (median 6.01, 99th percentile 19.01, maximum 1,056.04). None is before the purchase.
- 127 lines have a deadline before the approval time, and 239 lines are more than 30 days out.
- **Decision:** `limit_days_min` stays in the table and is marked "To be confirmed" in the dictionary. Whether it is known at approval cannot be proven from the data.

#### Step 3: sellers (of the primary seller)
**Question:** does every primary seller exist, and what is added?

In [10]:
assert sellers["seller_id"].is_unique

seller_cols = sellers.rename(columns={"seller_id": "primary_seller_id"})
before = len(base)
base = base.merge(seller_cols, on="primary_seller_id", how="left", validate="many_to_one")
matched = base["seller_state"].notna().sum()
log_join("3 sellers (primary seller)", "sellers", len(sellers), len(base), int(matched), "key primary_seller_id; orders without items stay empty")

print("rows before / after:", before, "/", len(base))
print("orders with a primary seller        :", base["primary_seller_id"].notna().sum())
print("... of these found in sellers       :", matched)
base[["order_id", "primary_seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]].dropna().head(3)

rows before / after: 99441 / 99441
orders with a primary seller        : 98666
... of these found in sellers       : 98666


,order_id,primary_seller_id,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,9350.0,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,289cdb325fb7e7f891c38608bf9e0962,31570.0,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,4869f7a5dfa277a7dca6462dcf3b52b2,14840.0,guariba,SP


- All 98,666 orders with items find their primary seller in the sellers table. The join is many-to-one and keeps 99,441 rows.
- The 775 orders without items have empty seller fields.

#### Step 4: payments
Payments have several lines per order, so they are reduced to one row per order first.

**Question 4a:** what does one row per order look like, and what does a direct merge cost?

In [11]:
seq = payments.groupby("order_id")["payment_sequential"].agg(["min", "max", "nunique", "size"])
seq_ok = (seq["min"] == 1) & (seq["max"] == seq["size"]) & (seq["nunique"] == seq["size"])

type_value = payments.groupby(["order_id", "payment_type"], as_index=False)["payment_value"].sum()
main_type = (type_value.sort_values(["order_id", "payment_value", "payment_type"], ascending=[True, False, True])
             .groupby("order_id").head(1).set_index("order_id")["payment_type"])

order_pay = payments.groupby("order_id").agg(
    n_payments=("payment_sequential", "size"),
    n_payment_types=("payment_type", "nunique"),
    total_paid=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
)
order_pay["main_payment_type"] = main_type
order_pay["seq_gap"] = ~seq_ok
assert order_pay.index.is_unique

naive_pay = orders.merge(payments, on="order_id", how="left")
print("direct merge orders + payment lines:", len(naive_pay), "-> extra rows:", len(naive_pay) - len(orders))

before = len(base)
base = base.merge(order_pay, left_on="order_id", right_index=True, how="left", validate="one_to_one")
base["has_payment"] = base["n_payments"].notna()
log_join("4 payments", "payments (reduced to one row per order)", len(order_pay), len(base), int(base["has_payment"].sum()), "key order_id")
print("rows before / after:", before, "/", len(base), "| orders with payments:", base["has_payment"].sum())

no_pay = base[~base["has_payment"]]
print("\norders without payment:", len(no_pay))
display(no_pay[["order_id", "order_status", "has_items", "order_approved_at"]])
add_check("payments", "orders without any payment line", len(no_pay), "status: " + ", ".join(no_pay["order_status"].astype(str)))
add_check("payments", "orders with a broken payment sequence (seq_gap)", int(base["seq_gap"].fillna(False).sum()))
add_check("payments", "rows after a direct merge of payment lines onto orders", len(naive_pay), f"{len(naive_pay) - len(orders)} extra rows")

direct merge orders + payment lines: 103887 -> extra rows: 4446
rows before / after: 99441 / 99441 | orders with payments: 99440

orders without payment: 1


,order_id,order_status,has_items,order_approved_at
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,True,2016-09-15 12:16:38


- A direct merge of payment lines gives 103,887 rows (4,446 more than the orders). After reducing to one row per order, the join keeps 99,441 rows, with 99,440 orders matched.
- One order has no payment line: a `delivered` order with items, approved on 2016-09-15 (the earliest approval in the data). It stays in the table with `has_payment` false.
- The 80 orders with a broken payment numbering are marked with `seq_gap`.

**Question 4b:** does the total paid match item price plus freight?

In [12]:
both = base[base["has_items"] & base["has_payment"]].copy()
both["items_total"] = both["total_price"] + both["total_freight"]
both["paid_minus_items"] = both["total_paid"] - both["items_total"]
d = both["paid_minus_items"]

print("orders with items and payments:", len(both))
print("paid within 0.01 of items total :", (d.abs() <= 0.01).sum(), f"({(d.abs() <= 0.01).mean() * 100:.1f}%)")
print("paid within 1.00 of items total :", (d.abs() <= 1.0).sum(), f"({(d.abs() <= 1.0).mean() * 100:.1f}%)")
print("paid MORE than items by over 1.00 :", (d > 1).sum(), f"({(d > 1).mean() * 100:.1f}%)")
print("paid LESS than items by over 1.00 :", (d < -1).sum(), f"({(d < -1).mean() * 100:.1f}%)")
display(d.describe(percentiles=[0.01, 0.5, 0.99]).round(2).to_frame("paid minus items total"))

# Are the differences linked to installments or to a broken sequence?
both["installment_group"] = pd.cut(both["max_installments"], [0, 1, 6, 100], labels=["1", "2-6", "7+"])
by_inst = both.groupby("installment_group", observed=True).agg(n_orders=("order_id", "size"),
                                                              paid_more_pct=("paid_minus_items", lambda s: (s > 1).mean() * 100),
                                                              median_diff=("paid_minus_items", "median")).round(2)
display(by_inst)
gap_orders = both[both["seq_gap"]]
print("orders with a broken payment sequence:", len(gap_orders),
      "| paid LESS than items by over 1.00:", (gap_orders["paid_minus_items"] < -1).sum())

add_check("payments", "paid within 0.01 of item price plus freight", f"{(d.abs() <= 0.01).mean() * 100:.1f}%", f"{len(both)} orders compared")
add_check("payments", "paid more than item price plus freight by over 1.00", f"{(d > 1).mean() * 100:.1f}%", "see installments")
add_check("payments", "paid less than item price plus freight by over 1.00", f"{(d < -1).mean() * 100:.1f}%")

orders with items and payments: 98665
paid within 0.01 of items total : 98284 (99.6%)
paid within 1.00 of items total : 98416 (99.7%)
paid MORE than items by over 1.00 : 232 (0.2%)
paid LESS than items by over 1.00 : 17 (0.0%)


,paid minus items total
count,98665.00
mean,0.03
std,1.13
min,-51.62
1%,-0.00
50%,0.00
99%,0.00
max,182.81


,n_orders,paid_more_pct,median_diff
installment_group,,,
1,47823,0.01,0.0
2-6,38759,0.34,0.0
7+,12081,0.81,0.0


orders with a broken payment sequence: 79 | paid LESS than items by over 1.00: 0


- For 98,665 orders with items and payments, the paid total equals item price plus freight (within 0.01) in 99.6% of cases, and within 1.00 in 99.7%.
- 232 orders (0.2%) paid more than 1.00 above the items total, and 17 paid more than 1.00 below.
- Paying more is rare for single-installment orders (0.01%) and rises with installments: 0.34% for 2 to 6 and 0.81% for 7 or more. This fits interest added to instalment plans, but the data does not say so.
- The 79 orders with a broken payment numbering that could be compared show no shortfall (0 paid less than the items total). The suspicion that payment 1 was missing and the amount was understated is not supported. The gap in numbering is a harmless label problem.

#### Step 5: geolocation lookup and distance
The geolocation table is first turned into one row per zip prefix (the rules come from the geolocation notebook), then joined to both the customer and the seller prefix.

**Question 5a:** what does the one-row-per-prefix lookup contain?

In [13]:
g = geo.drop_duplicates()                                             # no exact copies
in_brazil = g["geolocation_lat"].between(-33.75, 5.27) & g["geolocation_lng"].between(-73.99, -34.79)
valid = g[in_brazil]                                                  # points inside Brazil only

box = valid.groupby("geolocation_zip_code_prefix").agg(lat_min=("geolocation_lat", "min"), lat_max=("geolocation_lat", "max"),
                                                      lng_min=("geolocation_lng", "min"), lng_max=("geolocation_lng", "max"))
mid_lat = (box["lat_min"] + box["lat_max"]) / 2
box["spread_km"] = np.sqrt(((box["lat_max"] - box["lat_min"]) * 111) ** 2 +
                           ((box["lng_max"] - box["lng_min"]) * 111 * np.cos(np.radians(mid_lat))) ** 2)

geo_prefix = valid.groupby("geolocation_zip_code_prefix").agg(
    lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"),
    geo_state=("geolocation_state", lambda s: s.mode().iat[0]))
geo_prefix["spread_km"] = box["spread_km"]
geo_prefix["low_quality"] = geo_prefix["spread_km"] > 50
assert geo_prefix.index.is_unique

print("prefixes in the raw geolocation table:", geo["geolocation_zip_code_prefix"].nunique())
print("prefixes in the lookup               :", len(geo_prefix))
print("low quality prefixes (spread > 50 km):", geo_prefix["low_quality"].sum())
geo_prefix.head(3).round(4)

prefixes in the raw geolocation table: 19015
prefixes in the lookup               : 19010
low quality prefixes (spread > 50 km): 484


,lat,lng,geo_state,spread_km,low_quality
geolocation_zip_code_prefix,,,,,
1001,-23.5500,-46.6340,SP,0.2522,False
1002,-23.5482,-46.6352,SP,0.5710,False
1003,-23.5490,-46.6353,SP,0.2344,False


- The lookup has one row per prefix: 19,010 of the 19,015 raw prefixes (the other 5 have no valid point). 484 prefixes are flagged low quality (spread above 50 km).

**Question 5b:** can the customer and seller prefixes be located, and does the state match?

In [14]:
cust_ok = customers["customer_zip_code_prefix"].isin(geo_prefix.index)
sell_ok = sellers["seller_zip_code_prefix"].isin(geo_prefix.index)
print("customers whose prefix is in the lookup:", cust_ok.sum(), "of", len(customers), "| not found:", (~cust_ok).sum(),
      "| distinct prefixes not found:", customers.loc[~cust_ok, "customer_zip_code_prefix"].nunique())
print("sellers whose prefix is in the lookup  :", sell_ok.sum(), "of", len(sellers), "| not found:", (~sell_ok).sum(),
      "| distinct prefixes not found:", sellers.loc[~sell_ok, "seller_zip_code_prefix"].nunique())

# Does the state in the customers / sellers table agree with the state of the prefix?
cs = customers.merge(geo_prefix[["geo_state"]], left_on="customer_zip_code_prefix", right_index=True, how="inner", validate="many_to_one")
ss = sellers.merge(geo_prefix[["geo_state"]], left_on="seller_zip_code_prefix", right_index=True, how="inner", validate="many_to_one")
cust_mismatch = (cs["customer_state"] != cs["geo_state"]).sum()
sell_mismatch = (ss["seller_state"] != ss["geo_state"]).sum()
print("\ncustomers whose state differs from the prefix state:", cust_mismatch, f"({cust_mismatch / len(cs) * 100:.2f}% of matched)")
print("sellers whose state differs from the prefix state  :", sell_mismatch, f"({sell_mismatch / len(ss) * 100:.2f}% of matched)")
display(ss.loc[ss["seller_state"] != ss["geo_state"], ["seller_zip_code_prefix", "seller_city", "seller_state", "geo_state"]].head(5))

add_check("geolocation", "customers whose prefix is not in the lookup", int((~cust_ok).sum()))
add_check("geolocation", "sellers whose prefix is not in the lookup", int((~sell_ok).sum()))
add_check("geolocation", "customers whose state differs from the prefix state", int(cust_mismatch))
add_check("geolocation", "sellers whose state differs from the prefix state", int(sell_mismatch))

customers whose prefix is in the lookup: 99162 of 99441 | not found: 279 | distinct prefixes not found: 158
sellers whose prefix is in the lookup  : 3088 of 3095 | not found: 7 | distinct prefixes not found: 7

customers whose state differs from the prefix state: 0 (0.00% of matched)
sellers whose state differs from the prefix state  : 35 (1.13% of matched)


,seller_zip_code_prefix,seller_city,seller_state,geo_state
70,95076,caxias do sul,SP,RS
114,21210,rio de janeiro,RN,RJ
198,85301,laranjeiras do sul,SP,PR
206,87360,goioere,SP,PR
275,86170,sertanopolis,SP,PR


- 279 customers (158 prefixes) and 7 sellers (7 prefixes) have a prefix that is not in the lookup. This is 0.28% of customers and 0.23% of sellers.
- The customer state always equals the state of its prefix (0 differences). 35 sellers (1.13% of matched) have a state that differs from their prefix. In the examples the seller state is `SP` or `RN` while the city and prefix point to RS, PR or RJ, so the seller state is the faulty field.
- **Decision:** keep `seller_state` as given, and prefer the zip prefix for any location feature.

**Question 5c:** what is the distance between seller and customer for each order?

In [15]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance in km between two points (degrees)."""
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin((np.radians(lng2) - np.radians(lng1)) / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


cust_geo = geo_prefix[["lat", "lng", "low_quality"]].add_prefix("customer_")
sell_geo = geo_prefix[["lat", "lng", "low_quality"]].add_prefix("seller_")

before = len(base)
base = base.merge(cust_geo, left_on="customer_zip_code_prefix", right_index=True, how="left", validate="many_to_one")
log_join("5a geolocation (customer prefix)", "geolocation lookup", len(geo_prefix), len(base), int(base["customer_lat"].notna().sum()),
         "key customer_zip_code_prefix")
base = base.merge(sell_geo, left_on="seller_zip_code_prefix", right_index=True, how="left", validate="many_to_one")
log_join("5b geolocation (seller prefix)", "geolocation lookup", len(geo_prefix), len(base), int(base["seller_lat"].notna().sum()),
         "key seller_zip_code_prefix; orders without items stay empty")

base["distance_km"] = haversine_km(base["customer_lat"], base["customer_lng"], base["seller_lat"], base["seller_lng"])
base["distance_low_quality"] = base["customer_low_quality"].fillna(False).astype(bool) | base["seller_low_quality"].fillna(False).astype(bool)
base["geo_missing"] = base["distance_km"].isna()

has_seller = base["has_items"]
print("rows before / after:", before, "/", len(base))
print("orders with items            :", has_seller.sum())
print("... with a distance          :", base.loc[has_seller, "distance_km"].notna().sum())
print("... without a distance       :", base.loc[has_seller, "distance_km"].isna().sum())
print("orders with a low-quality prefix on either side:", base["distance_low_quality"].sum())
display(base["distance_km"].describe(percentiles=[0.05, 0.5, 0.95, 0.99]).round(1).to_frame("distance_km"))
print("distance above 3000 km:", (base["distance_km"] > 3000).sum())
add_check("geolocation", "orders with items but no distance", int(base.loc[has_seller, "distance_km"].isna().sum()))
add_check("geolocation", "orders with a low-quality prefix on either side", int(base["distance_low_quality"].sum()))
add_check("geolocation", "median distance seller to customer (km)", round(float(base["distance_km"].median()), 1))

rows before / after: 99441 / 99441
orders with items            : 98666
... with a distance          : 98176
... without a distance       : 490
orders with a low-quality prefix on either side: 4692


,distance_km
count,98176.0
mean,601.3
std,593.9
min,0.0
5%,16.5
50%,433.8
95%,2098.4
99%,2482.8
max,3579.3


distance above 3000 km: 136


- Of the 98,666 orders with items, 98,176 have a distance and 490 do not (a prefix is missing from the lookup). They stay in the table with `geo_missing` true.
- The median seller-to-customer distance is 433.8 km, the 95th percentile 2,098 km and the maximum 3,579 km. 136 orders are above 3,000 km.
- 4,692 orders have a low-quality prefix on at least one side (`distance_low_quality`), so their distance is less reliable.
- Distance is known at checkout, so it can be a feature in Part 2.

#### Step 6: reviews (the target)
Reviews are reduced to one row per order (the latest review), and are used as the target only.

**Question:** how many orders have a review, and what is the share of bad orders?

In [16]:
review_count = reviews.groupby("order_id").size().rename("n_reviews")

# Latest review per order: creation date, then answer time, then review_id (a stable tie-break)
latest = (reviews.sort_values(["order_id", "review_creation_date", "review_answer_timestamp", "review_id"])
          .groupby("order_id").tail(1).set_index("order_id"))
latest["review_comment_message"] = latest["review_comment_message"].str.strip().replace("", np.nan)
order_rev = latest[["review_score", "review_creation_date", "review_answer_timestamp", "review_comment_message"]].join(review_count)
assert order_rev.index.is_unique

naive_rev = orders.merge(reviews, on="order_id", how="left")
print("direct merge orders + reviews:", len(naive_rev), "-> extra rows:", len(naive_rev) - len(orders))

before = len(base)
base = base.merge(order_rev, left_on="order_id", right_index=True, how="left", validate="one_to_one")
base["bad_review"] = np.where(base["review_score"].isna(), np.nan, base["review_score"].isin([1, 2]).astype(float))
log_join("6 reviews (latest per order)", "reviews (reduced to one row per order)", len(order_rev), len(base),
         int(base["review_score"].notna().sum()), "key order_id; target only")

print("rows before / after:", before, "/", len(base))
print("orders with a review   :", base["review_score"].notna().sum())
print("orders without a review:", base["review_score"].isna().sum())
print("bad orders (latest review 1 or 2):", int(base["bad_review"].sum()), f"({base['bad_review'].mean() * 100:.1f}% of reviewed orders)")
display(base.loc[base["review_score"].isna(), "order_status"].value_counts().rename("orders without a review").to_frame())

add_check("reviews", "orders without a review", int(base["review_score"].isna().sum()))
add_check("reviews", "bad share among reviewed orders (latest review)", f"{base['bad_review'].mean() * 100:.1f}%")
add_check("reviews", "rows after a direct merge of review rows onto orders", len(naive_rev), f"{len(naive_rev) - len(orders)} extra rows")

direct merge orders + reviews: 99992 -> extra rows: 551


rows before / after: 99441 / 99441
orders with a review   : 98673
orders without a review: 768
bad orders (latest review 1 or 2): 14495 (14.7% of reviewed orders)


,orders without a review
order_status,
delivered,646
shipped,75
canceled,20
unavailable,14
processing,6
invoiced,5
created,2


- A direct merge of review rows gives 99,992 rows (551 extra). Reducing to the latest review per order keeps 99,441 rows, with 98,673 orders matched.
- 768 orders have no review: 646 are `delivered` and 122 have another status. They get an empty target.
- 14,495 of the 98,673 reviewed orders are bad, which is 14.7%.
- The review fields are the target and are not features.

**Question:** how do review dates relate to the order dates?

In [17]:
rv = base.dropna(subset=["review_creation_date"]).copy()
# The review creation date is a day (midnight), so compare it with the purchase DAY, not the purchase time
purchase_to_review = (rv["review_creation_date"] - rv["order_purchase_timestamp"].dt.normalize()).dt.total_seconds() / 86400
print("days from purchase to review creation: median", round(purchase_to_review.median(), 1),
      "| 5th pct", round(purchase_to_review.quantile(0.05), 1), "| 95th pct", round(purchase_to_review.quantile(0.95), 1))
print("reviews created before the purchase day           :", (purchase_to_review < 0).sum())
print("reviews created on the purchase day               :", (purchase_to_review == 0).sum())
delivered = rv.dropna(subset=["order_delivered_customer_date"])
print("reviews created BEFORE the delivery date          :", (delivered["review_creation_date"] < delivered["order_delivered_customer_date"].dt.normalize()).sum(),
      "of", len(delivered), "delivered orders with a review")
print("last review creation date / last purchase date    :", rv["review_creation_date"].max().date(), "/", base["order_purchase_timestamp"].max().date())

# Review month against purchase month (share of reviews in the same month)
same_month = (rv["review_creation_date"].dt.to_period("M") == rv["order_purchase_timestamp"].dt.to_period("M")).mean()
print("review created in the same month as the purchase  :", f"{same_month * 100:.1f}%")
add_check("reviews", "reviews created before the delivery date", int((delivered["review_creation_date"] < delivered["order_delivered_customer_date"].dt.normalize()).sum()))
add_check("reviews", "reviews created before the purchase day", int((purchase_to_review < 0).sum()))
add_check("reviews", "median days from purchase to review creation", round(float(purchase_to_review.median()), 1))

days from purchase to review creation: median 11.0 | 5th pct 4.0 | 95th pct 29.0
reviews created before the purchase day           : 61
reviews created on the purchase day               : 10
reviews created BEFORE the delivery date          : 4967 of 95830 delivered orders with a review
last review creation date / last purchase date    : 2018-08-31 / 2018-10-17
review created in the same month as the purchase  : 58.7%


- The median review is created 11.0 days after the purchase day (5th percentile 4.0, 95th percentile 29.0). 58.7% are created in the same month as the purchase.
- 61 reviews are created before the purchase day, which is impossible, and 10 on the purchase day. They are kept, and the 61 point to a date error.
- 4,967 of 95,830 delivered orders with a review (5.2%) have a review created on an earlier day than the delivery date. So a review can come before the delivery is recorded, and the target is not always created after delivery.
- The last review was created on 2018-08-31, while purchases run to 2018-10-17. Orders bought after the end of August 2018 cannot have a review in this data, which is why the modelling window ends in Aug 2018.

#### Derived columns and scope flags
**Question:** which timing columns and flags are added? No row is dropped here.

In [18]:
DAY = 86400
base["approve_h"] = (base["order_approved_at"] - base["order_purchase_timestamp"]).dt.total_seconds() / 3600       # known at approval
base["promised_d"] = (base["order_estimated_delivery_date"] - base["order_purchase_timestamp"]).dt.total_seconds() / DAY   # known at checkout
base["total_d"] = (base["order_delivered_customer_date"] - base["order_purchase_timestamp"]).dt.total_seconds() / DAY      # NOT known
base["delay_d"] = (base["order_delivered_customer_date"] - base["order_estimated_delivery_date"]).dt.total_seconds() / DAY  # NOT known

# Contradictions between status and delivery date (from the orders notebook)
base["status_conflict"] = ((base["order_status"] == "delivered") & base["order_delivered_customer_date"].isna()) | \
                          ((base["order_status"] == "canceled") & base["order_delivered_customer_date"].notna())

# Scope flags (proposals for Part 2). Rows are kept, flags only mark them.
WINDOW = (pd.Timestamp("2017-01-01"), pd.Timestamp("2018-08-31 23:59:59"))
base["flag_not_approved"] = base["order_approved_at"].isna()
base["flag_no_items"] = ~base["has_items"]
base["flag_outside_window"] = ~base["order_purchase_timestamp"].between(*WINDOW)
base["in_scope_proposal"] = ~(base["flag_not_approved"] | base["flag_no_items"] | base["flag_outside_window"])

flags = pd.DataFrame({"n_orders": [base["flag_not_approved"].sum(), base["flag_no_items"].sum(), base["flag_outside_window"].sum(),
                                   base["status_conflict"].sum(), base["in_scope_proposal"].sum()]},
                     index=["never approved", "no items", "purchase outside 2017-01-01 to 2018-08-31", "status conflict (not a scope flag)", "IN SCOPE (proposal)"])
flags["pct_of_orders"] = (flags["n_orders"] / len(base) * 100).round(2)
display(flags)

scope = base[base["in_scope_proposal"]]
print("bad share among reviewed orders, in scope:", f"{scope['bad_review'].mean() * 100:.1f}%", "| orders:", len(scope))
add_check("scope", "orders in scope (proposal: approved, has items, 2017-01 to 2018-08)", int(base["in_scope_proposal"].sum()))
add_check("scope", "bad share among reviewed orders in scope", f"{scope['bad_review'].mean() * 100:.1f}%")

,n_orders,pct_of_orders
never approved,160,0.16
no items,775,0.78
purchase outside 2017-01-01 to 2018-08-31,349,0.35
status conflict (not a scope flag),14,0.01
IN SCOPE (proposal),98339,98.89


bad share among reviewed orders, in scope: 14.2% | orders: 98339


- 160 orders were never approved, 775 have no items, and 349 were purchased outside 2017-01-01 to 2018-08-31 (the thin 2016 start and the Sep and Oct 2018 end seen in the orders notebook). 14 orders have a status conflict.
- 98,339 orders (98.89%) are in scope under this proposal. The bad share in scope is 14.2%, a little below the 14.7% of all reviewed orders.
- No row is dropped. The flags only mark rows, so the scope can be changed in Part 2 without rebuilding the table.

#### Missingness vs the target
**Question:** which missing or flagged fields relate to bad reviews? The share is the bad-review rate among reviewed orders.

In [19]:
conditions = {
    "approval time missing":            base["order_approved_at"].isna(),
    "carrier date missing":             base["order_delivered_carrier_date"].isna(),
    "delivery date missing":            base["order_delivered_customer_date"].isna(),
    "no item lines":                    ~base["has_items"],
    "no payment line":                  ~base["has_payment"],
    "product without listing info":     base["any_no_listing_info"].fillna(False).astype(bool),
    "weight incomplete":                base["weight_incomplete"].fillna(False).astype(bool),
    "no distance (geo missing)":        base["geo_missing"] & base["has_items"],
    "low-quality prefix":               base["distance_low_quality"],
    "broken payment sequence":          base["seq_gap"].fillna(False).astype(bool),
}
rows = []
for name, cond in conditions.items():
    cond = cond.astype(bool)
    rows.append({"condition": name, "n_orders": int(cond.sum()),
                 "bad_pct_if_true": round(base.loc[cond, "bad_review"].mean() * 100, 1),
                 "bad_pct_if_false": round(base.loc[~cond, "bad_review"].mean() * 100, 1)})
miss_vs_target = pd.DataFrame(rows).set_index("condition")
display(miss_vs_target)
for name, r in miss_vs_target.iterrows():
    add_check("missingness vs target", name, f"bad {r['bad_pct_if_true']}% if true vs {r['bad_pct_if_false']}% if false", f"{int(r['n_orders'])} orders")

,n_orders,bad_pct_if_true,bad_pct_if_false
condition,,,
approval time missing,160,51.0,14.6
carrier date missing,1783,83.2,13.5
delivery date missing,2965,78.1,12.8
no item lines,775,79.5,14.2
no payment line,1,100.0,14.7
product without listing info,1451,20.2,14.6
weight incomplete,16,18.8,14.7
no distance (geo missing),490,13.3,14.7
low-quality prefix,4692,15.5,14.6


- The strongest links come from post-approval gaps: delivery date missing 78.1% bad against 12.8%, carrier date missing 83.2% against 13.5%, approval time missing 51.0% against 14.6%, and no item lines 79.5% against 14.2%. These are known only after approval (or never), so none can be a feature.
- One missing field is known at approval and still relates to the outcome: products without listing information (1,451 orders) have 20.2% bad against 14.6%.
- Smaller signals: a broken payment numbering 20.3% (80 orders) and a low-quality location prefix 15.5% (4,692 orders).
- Not informative: a missing distance (13.3%, 490 orders). Too small to read: no payment line (1 order) and incomplete weight (16 orders).
- Reading for Part 2: the missingness tied to the target is the lifecycle missingness (the event never happened). Training only on delivered orders would remove the worst cases.

#### Data dictionary
**Question:** is every column described, and is it known at order approval? This is the guard against leakage in Part 2.

In [20]:
D = [  # column, source, description, known at approval?
 ("order_id", "orders", "Order key (one row per order)", "Key"),
 ("customer_id", "orders", "Per-order customer id, links to customers", "Key"),
 ("order_status", "orders", "Final status of the order", "No"),
 ("order_purchase_timestamp", "orders", "When the order was placed", "Yes"),
 ("order_approved_at", "orders", "When the order was approved (the prediction moment)", "Prediction moment"),
 ("order_delivered_carrier_date", "orders", "When the carrier received the order", "No"),
 ("order_delivered_customer_date", "orders", "When the customer received the order", "No"),
 ("order_estimated_delivery_date", "orders", "Delivery date promised at checkout", "Yes"),
 ("customer_unique_id", "customers", "Person id (one person can have several customer_id values)", "Yes"),
 ("customer_zip_code_prefix", "customers", "Customer zip prefix (number, leading zero lost)", "Yes"),
 ("customer_city", "customers", "Customer city text", "Yes"),
 ("customer_state", "customers", "Customer state", "Yes"),
 ("n_lines", "items", "Number of item lines in the order", "Yes"),
 ("n_products", "items", "Number of distinct products", "Yes"),
 ("n_sellers", "items", "Number of distinct sellers", "Yes"),
 ("n_categories", "items + products", "Number of distinct categories (known ones)", "Yes"),
 ("total_price", "items", "Sum of item prices", "Yes"),
 ("total_freight", "items", "Sum of freight values", "Yes"),
 ("total_weight_g", "products", "Sum of declared product weights (empty if all missing)", "Yes"),
 ("total_volume_cm3", "products", "Sum of declared box volumes", "Yes"),
 ("weight_incomplete", "products", "At least one line has no declared weight", "Yes"),
 ("any_no_listing_info", "products", "At least one product has no listing information", "Yes"),
 ("limit_days_min", "items", "Earliest shipping deadline, days after purchase", "To be confirmed"),
 ("freight_ratio", "items", "total_freight divided by total_price", "Yes"),
 ("primary_seller_id", "items", "Seller with the largest share of the order value", "Yes"),
 ("primary_category", "items + products", "Category with the largest share of the order value (Portuguese)", "Yes"),
 ("primary_category_en", "category translation", "English name of the primary category (5 spelling errors fixed, empty if no translation)", "Yes"),
 ("has_items", "items", "Order has at least one item line", "To be confirmed"),
 ("seller_zip_code_prefix", "sellers", "Primary seller zip prefix", "Yes"),
 ("seller_city", "sellers", "Primary seller city text (dirty)", "Yes"),
 ("seller_state", "sellers", "Primary seller state (less reliable than the prefix)", "Yes"),
 ("n_payments", "payments", "Number of payment lines", "Probably"),
 ("n_payment_types", "payments", "Number of distinct payment types", "Yes"),
 ("total_paid", "payments", "Sum of payment values", "Yes"),
 ("max_installments", "payments", "Highest number of installments", "Yes"),
 ("main_payment_type", "payments", "Payment type with the largest paid value", "Yes"),
 ("seq_gap", "payments", "Payment numbers do not run 1..n (first payment may be missing)", "Yes (data-quality flag)"),
 ("has_payment", "payments", "Order has at least one payment line", "Yes"),
 ("customer_lat", "geolocation lookup", "Median latitude of the customer prefix", "Yes"),
 ("customer_lng", "geolocation lookup", "Median longitude of the customer prefix", "Yes"),
 ("customer_low_quality", "geolocation lookup", "Customer prefix points spread over more than 50 km", "Yes"),
 ("seller_lat", "geolocation lookup", "Median latitude of the seller prefix", "Yes"),
 ("seller_lng", "geolocation lookup", "Median longitude of the seller prefix", "Yes"),
 ("seller_low_quality", "geolocation lookup", "Seller prefix points spread over more than 50 km", "Yes"),
 ("distance_km", "geolocation lookup", "Great-circle distance between seller and customer prefix", "Yes"),
 ("distance_low_quality", "geolocation lookup", "Either prefix has a low-quality location", "Yes"),
 ("geo_missing", "geolocation lookup", "No distance could be computed", "Yes"),
 ("n_reviews", "reviews", "Number of reviews of the order", "No"),
 ("review_score", "reviews", "Score of the latest review (1 to 5)", "Target"),
 ("review_creation_date", "reviews", "Creation date of the latest review", "No"),
 ("review_answer_timestamp", "reviews", "Answer time of the latest review", "No"),
 ("review_comment_message", "reviews", "Comment text of the latest review (Portuguese)", "No (used in Part 4)"),
 ("bad_review", "reviews", "1 if the latest score is 1 or 2, 0 if 3 to 5, empty if no review", "Target"),
 ("approve_h", "derived", "Hours from purchase to approval", "Yes"),
 ("promised_d", "derived", "Days from purchase to the estimated delivery date", "Yes"),
 ("total_d", "derived", "Days from purchase to delivery", "No"),
 ("delay_d", "derived", "Days after the estimated date (negative = early)", "No"),
 ("status_conflict", "derived", "Status and delivery date contradict each other", "No"),
 ("flag_not_approved", "scope flag", "Order was never approved (no prediction moment)", "Flag"),
 ("flag_no_items", "scope flag", "Order has no item lines", "Flag"),
 ("flag_outside_window", "scope flag", "Purchase outside 2017-01-01 to 2018-08-31", "Flag"),
 ("in_scope_proposal", "scope flag", "None of the three scope flags is set (proposal, not applied)", "Flag"),
]
dictionary = pd.DataFrame(D, columns=["column", "source", "description", "known_at_approval"])

# Every column of the final table must be described, and every description must match a column
final_cols = [c for c in dictionary["column"] if c in base.columns]
missing_in_dict = [c for c in base.columns if c not in set(dictionary["column"])]
missing_in_table = [c for c in dictionary["column"] if c not in base.columns]
print("columns in the table        :", base.shape[1])
print("columns in the dictionary   :", len(dictionary))
print("table columns not described :", missing_in_dict)
print("described but not in table  :", missing_in_table)
assert not missing_in_dict and not missing_in_table

base = base[list(dictionary["column"])]            # same order as the dictionary
dictionary["known_at_approval"].value_counts().rename("n_columns").to_frame()

columns in the table        : 62
columns in the dictionary   : 62
table columns not described : []
described but not in table  : []


,n_columns
known_at_approval,
Yes,39
No,9
Flag,4
Key,2
To be confirmed,2
Target,2
Prediction moment,1
Probably,1
Yes (data-quality flag),1


- All 62 columns are described, and every description matches a column.
- 39 columns are marked `Yes` (known at approval). Another `Yes` is a data-quality flag, `n_payments` is `Probably`, and `limit_days_min` and `has_items` are `To be confirmed`.
- 9 columns are `No`, 1 more is `No (used in Part 4)`, 2 are the target, 2 are keys, 4 are scope flags and 1 is the prediction moment itself.
- In Part 2 the feature list can be taken straight from this sheet. Anything marked No, Target or Flag stays out.

#### Save to Excel
**Question:** is the joined table saved in one Excel file, with the dictionary, the join log and the checks?

Excel handles this size (99,441 rows is far below the limit of 1,048,576). The file is written once with five sheets.

In [21]:
read_me = pd.DataFrame([
    ("What this file is", "Joined order-level table of the Olist dataset. One row per order."),
    ("Rows", f"{len(base):,} orders. Grain proof: every join was checked to keep this count (see sheet join_log)."),
    ("Built by", "notebooks/eda/eda_10_joined.ipynb from the raw Kaggle CSV files (not included, not redistributed)."),
    ("Sheets", "read_me | orders_joined (data) | data_dictionary (every column, source, known at approval) | join_log | checks"),
    ("Target", "bad_review = 1 if the latest review of the order is 1 or 2 stars. Empty if the order has no review."),
    ("Leakage guard", "Only columns marked Yes (or Prediction moment) in data_dictionary may be used as features. No, Target and Flag columns may not."),
    ("Scope flags", "flag_* columns and in_scope_proposal only mark orders. No row was dropped. The scope is a proposal for Part 2."),
    ("Seller and category", "primary_seller_id and primary_category = the seller or category with the largest share of the order value."),
    ("Empty cells", "Empty means the event did not happen or the value is unknown. Nothing was filled or imputed."),
    ("Zip prefixes", "Stored as numbers, so a leading zero is lost (for example 1046 means 01046)."),
], columns=["item", "note"])

PATH = OUT_DIR / "orders_joined.xlsx"
join_log_df = pd.DataFrame(join_log)
checks_df = pd.DataFrame(checks)

# Arial 10 is the default font of every format the writer creates (including the date format)
with pd.ExcelWriter(PATH, engine="xlsxwriter", datetime_format="yyyy-mm-dd hh:mm:ss",
                    engine_kwargs={"options": {"default_format_properties": {"font_name": "Arial", "font_size": 10}}}) as writer:
    wb = writer.book
    head = wb.add_format({"bold": True, "font_name": "Arial", "font_size": 10, "bg_color": "#D9E1F2", "border": 1, "valign": "top", "text_wrap": True})
    wrap = wb.add_format({"font_name": "Arial", "font_size": 10, "text_wrap": True, "valign": "top"})
    money = wb.add_format({"font_name": "Arial", "font_size": 10, "num_format": "#,##0.00"})
    ratio = wb.add_format({"font_name": "Arial", "font_size": 10, "num_format": "0.00"})
    plain = wb.add_format({"font_name": "Arial", "font_size": 10})

    def write_sheet(df, name, widths=None):
        df.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        for j, col in enumerate(df.columns):
            ws.write(0, j, col, head)
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, len(df), len(df.columns) - 1)
        return ws

    # read_me
    ws = write_sheet(read_me, "read_me")
    ws.set_column(0, 0, 24, wrap)
    ws.set_column(1, 1, 120, wrap)

    # data sheet
    ws = write_sheet(base, "orders_joined")
    money_cols = {"total_price", "total_freight", "total_paid"}
    ratio_cols = {"freight_ratio", "approve_h", "promised_d", "total_d", "delay_d", "distance_km", "limit_days_min",
                  "customer_lat", "customer_lng", "seller_lat", "seller_lng"}
    for j, col in enumerate(base.columns):
        if col in money_cols:
            ws.set_column(j, j, 14, money)
        elif col in ratio_cols:
            ws.set_column(j, j, 14, ratio)
        elif str(base[col].dtype).startswith("datetime"):
            ws.set_column(j, j, 20, plain)
        elif col in {"order_id", "customer_id", "customer_unique_id", "primary_seller_id"}:
            ws.set_column(j, j, 34, plain)
        elif col == "review_comment_message":
            ws.set_column(j, j, 50, plain)
        else:
            ws.set_column(j, j, max(14, min(26, len(col) + 2)), plain)

    # dictionary, join log, checks
    ws = write_sheet(dictionary, "data_dictionary")
    ws.set_column(0, 0, 28, plain); ws.set_column(1, 1, 22, plain); ws.set_column(2, 2, 80, wrap); ws.set_column(3, 3, 24, plain)
    ws = write_sheet(join_log_df, "join_log")
    ws.set_column(0, 0, 34, plain); ws.set_column(1, 1, 40, plain); ws.set_column(2, 5, 16, plain); ws.set_column(6, 6, 70, wrap)
    ws = write_sheet(checks_df, "checks")
    ws.set_column(0, 0, 22, plain); ws.set_column(1, 1, 70, wrap); ws.set_column(2, 2, 20, plain); ws.set_column(3, 3, 60, wrap)

size_mb = PATH.stat().st_size / 1024 / 1024
print("saved:", PATH, f"({size_mb:.1f} MB)")

# Read it back to prove the file is complete
check = pd.read_excel(PATH, sheet_name=None)
print({name: df.shape for name, df in check.items()})
assert check["orders_joined"].shape == base.shape
assert check["orders_joined"]["order_id"].is_unique

saved: ../../data/processed/orders_joined.xlsx (45.4 MB)


{'read_me': (10, 2), 'orders_joined': (99441, 62), 'data_dictionary': (62, 4), 'join_log': (7, 7), 'checks': (43, 4)}


- The Excel file has 5 sheets: `read_me`, `orders_joined` (99,441 rows and 62 columns), `data_dictionary`, `join_log` and `checks`. It was read back, and the shape and the unique `order_id` were verified.
- The file is about 45 MB, so opening it takes a little time, but 99,441 rows are far below the Excel limit of 1,048,576.
- Dates are written as real Excel dates, empty cells are blank, and the font is Arial throughout.

#### Summary

In [22]:
display(join_log_df.set_index("step"))
checks_df.groupby("area").size().rename("n_checks").to_frame()

,right_table,right_rows_used,rows_after_join,matched_orders,unmatched_orders,note
step,,,,,,
1 customers,customers,99441,99441,99441,0,"key customer_id, one to one"
2 items + products,items (reduced to one row per order),98666,99441,98666,775,key order_id; adds seller and category through...
3 sellers (primary seller),sellers,3095,99441,98666,775,key primary_seller_id; orders without items st...
4 payments,payments (reduced to one row per order),99440,99441,99440,1,key order_id
5a geolocation (customer prefix),geolocation lookup,19010,99441,99162,279,key customer_zip_code_prefix
5b geolocation (seller prefix),geolocation lookup,19010,99441,98450,991,key seller_zip_code_prefix; orders without ite...
6 reviews (latest per order),reviews (reduced to one row per order),98673,99441,98673,768,key order_id; target only


,n_checks
area,
geolocation,7
items,9
missingness vs target,10
payments,6
products,3
reviews,6
scope,2


**The story in short.** The 8 tables join into one table with one row per order (99,441). Every join keeps the row count. Items, payments and reviews had to be reduced to one row per order first, because direct merges would add 13,984, 4,446 and 551 rows. The cross-table checks from the earlier notebooks are answered in the sheet `checks`. The joined data is clean at the key level, but the target is tied to lifecycle gaps that are unknown at approval, which is the main design constraint for Part 2.

**Final data quality issues of the joined table**

| # | Issue | Records | Action | Cost of the alternative |
|---|---|---|---|---|
| 1 | Orders without item lines | 775 (79.5% bad) | Keep, `flag_no_items`, outside the scope proposal | A has-items feature may leak the outcome, and dropping hides the worst group |
| 2 | Orders never approved | 160 | Keep, `flag_not_approved`, no prediction moment | Imputing an approval time invents one |
| 3 | Purchases outside 2017-01 to 2018-08 | 349 | Keep, `flag_outside_window` | Using them mixes thin or unfinished months |
| 4 | Delivery and carrier dates missing | 2,965 and 1,783 | Keep empty, never a feature | Filling invents deliveries |
| 5 | Status and delivery date contradict | 14 | Keep, `status_conflict` | Guessing the true value |
| 6 | One delivered order without payment | 1 | Keep, `has_payment` false | Dropping a real order |
| 7 | Broken payment numbering | 80 | Keep, `seq_gap`. Totals still match, so low priority. | Dropping real orders |
| 8 | Paid total differs from items total | 232 more, 17 less (over 1.00) | Keep both columns | Choosing one hides the difference |
| 9 | Seller state differs from its prefix | 35 sellers | Keep, prefer the zip prefix | Trusting the state gives wrong regions |
| 10 | Prefix not found, or low quality | 279 customers and 7 sellers not found, 4,692 orders low quality | Keep, `geo_missing`, `distance_low_quality` | Dropping loses orders, a mean location is wrong |
| 11 | Orders without a review | 768 (646 delivered) | Keep with an empty target | Treating them as good invents a label |
| 12 | Several reviews per order | 547 orders | Label = latest review | Lowest review marks 38 more orders bad |
| 13 | Review before the purchase day | 61 | Keep, date error | Dropping loses the review |
| 14 | Review before the delivery day | 4,967 | Keep, note that the target is not always after delivery | Hides a timing fact |
| 15 | Products without listing info | 1,451 orders (20.2% bad) | Keep, `any_no_listing_info` | Imputing hides a real signal |

**Decisions to log in `DECISIONS.md`**
- Grain: one row per order, with a hard check after every join.
- Items, payments and reviews are reduced to one row per order before joining.
- Label: latest review per order. Alternative: lowest review.
- Primary seller and primary category: the largest share of the order value.
- Scope proposal: approved, has items, purchased 2017-01-01 to 2018-08-31. Flags only, no row dropped.
- Missing dates are kept empty. Nothing is imputed.
- Seller state is kept as given, and the zip prefix is preferred for location.
- English category names: 5 spelling errors fixed with an explicit list. 1 suspected wrong translation (`seguros_e_servicos`) is flagged and not changed.
- Geolocation: one row per prefix from the median of points inside Brazil, with a low-quality flag.

**Next:** Part 2 starts from this table and the dictionary sheet, with the as-of seller history (built only from orders that closed before the scored order), a time-based split, and the capacity-constrained threshold.